<!-- [c2-01-a] -->
# C2-01 — Retrieval concepts (themes A, B, C)

Re-derived from IBM's *Summarize Private Documents Using RAG, LangChain, and LLMs*
(`clones/IBM-RAG-and-Agentic-AI/02 Build RAG Applications/`). The original stays untouched —
see `C2-analysis.md` for the source-material convention.

**This notebook deliberately does not re-teach the RAG pipeline.** Loading, splitting, embedding,
storing and retrieving were built the hard way in `Demo_C1_project.ipynb`, with a
`ParentDocumentRetriever` on top. Section 0 below hands you that pipeline complete, on purpose.

What's left are the three ideas the original raises and never examines — one per theme:

| theme | the question |
|---|---|
| **A** | You retrieved N chunks. How do they become *one* answer? |
| **B** | "What can't I do in **it**?" — a vector search has no idea what "it" is. |
| **C** | The notebook is called *Summarize private documents*. Can this technique actually summarize a document? |

Corpus is IBM's `companyPolicies.txt` rather than the C1 Maquiavel PDF, for one reason: it's short
enough to read end to end, so when a summary silently drops half the document you can *see* what's
missing. (The Maquiavel corpus comes back later, as the golden set for the eval harness.)

python utils/nbtag.py NOTEBOOK            # dry run, prints the map <br>
python utils/nbtag.py NOTEBOOK --apply    # write/refresh tags.   <br>
python utils/nbtag.py NOTEBOOK --list     # read tags, change nothing


<!-- [setup-a] -->
## Setup

In [1]:
# [setup-b]
import os
import warnings

import wget
from dotenv import load_dotenv

# Legacy chains live in `langchain_classic` under langchain 1.x — see CLAUDE.md.
# They are the subject of study here, not a recommendation.
from langchain_classic.chains import ConversationalRetrievalChain, RetrievalQA
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_classic.memory import ConversationBufferMemory
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_core.callbacks import BaseCallbackHandler
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

warnings.filterwarnings("ignore")
load_dotenv()

DATA_DIR = "../../data"
CORPUS = f"{DATA_DIR}/companyPolicies.txt"
URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt"
)

if not os.path.exists(CORPUS):
    wget.download(URL, out=CORPUS)
print("corpus:", CORPUS, os.path.getsize(CORPUS), "bytes")

/var/folders/pf/7lwsqjw92g96dl5sfdckf9_r0000gn/T/ipykernel_3870/360917113.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


corpus: ../../data/companyPolicies.txt 15660 bytes


In [2]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

<!-- [0a] -->
## 0. Baseline pipeline — mostly given on purpose

The model, the loader and the splitter below are all things you built in C1. They are handed over
complete so this notebook can be about the three questions, not about re-typing a splitter.

The vector store is **not** handed over, for a reason you'll see in 0.1.

In [3]:
# [0b]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5, max_tokens=512)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

documents = TextLoader(CORPUS).load()
chunks = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0).split_documents(documents)

print(f"{len(chunks)} chunks")

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16 chunks


<!-- [0.1a] -->
### 0.1 — a trap, sprung before you trust any number in this notebook

Themes A and C both end in a *count*: how many LLM calls, how many policies covered. A count is
only worth something if the store underneath it holds what you think it holds.

Run the next cell. It builds the same five documents twice — the way section 0 would have, had it
been handed to you.

In [4]:
# [0.1b] Given: the trap. Read the two numbers before reading the explanation below.
from langchain_core.documents import Document
from langchain_core.embeddings import FakeEmbeddings

_demo_docs = [Document(page_content=f"policy {i}") for i in range(5)]
_demo = Chroma.from_documents(_demo_docs, FakeEmbeddings(size=8), collection_name="c2_trap_demo")
print("after 1st build:", _demo._collection.count())

_demo2 = Chroma.from_documents(_demo_docs, FakeEmbeddings(size=8), collection_name="c2_trap_demo")
print("after 2nd build:", _demo2._collection.count())
print("and the FIRST handle now sees:", _demo._collection.count())

_demo.delete_collection()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


after 1st build: 5
after 2nd build: 10
and the FIRST handle now sees: 10


<!-- [0.1c] -->
Five documents became ten. `from_documents` **appends**; it does not replace. Both handles point at
one collection in a client shared across the whole kernel, so re-running a build cell silently
doubles the corpus — and `k=4` starts returning two distinct chunks plus their copies.

Nothing errors. The notebook just quietly starts lying to you.

**This is the notebook's missing reset**, and the fix is not one rule but two, because these are
different kinds of state:

| | vector store | conversation memory (theme B) |
|---|---|---|
| what it is | expensive derived data | live state of one dialogue |
| **the rule** | **cache** — a second run must change nothing | **reset** — a second run must start from turn zero |
| symptom when the rule is broken | duplicate chunks, corrupted counts, embeddings paid for twice | polluted history, nonsense condensed questions |
| where you enforce it | `build_docsearch` — `[0.2b]` | `fresh_memory` — `[B.2b]` |

Cache the first. Reset the second. Swap them and notebooks become haunted.

**Read the rule row as a specification, not as a description of the cell above it.** Re-running is
an action, not a rule — ⇧⏎ twice, nothing more. It has no correct outcome of its own; the outcome
is whichever one you built. The cell above is deliberately the broken version, which is why
re-running it duplicates instead of doing nothing. **Nothing in this notebook satisfies the rule
yet** — 0.2 is where you make it true, and the assert there is what proves it.

One more distinction, because "failed to cache" has two flavours and `from_documents` picks the
worse one:

- **rebuild, replacing** — the count stays right; you only re-pay for the embeddings.
- **rebuild, appending** — the count goes wrong *and* you re-pay. ← what you just watched happen.

> **This is theme D arriving early.** That table is the two-lifetime problem from the application
> layer: an index cached *per document* versus a conversation reset *per thread* — icebreaker's
> `active_indices[session_id]` and your C1 checkpointer. In a notebook a mix-up costs you a wrong
> number. In a server it costs one user another user's document.


<!-- [0.2a] -->
### 0.2 — build the store so re-running is a no-op

Idempotent here means *build once, reuse after* — not *rebuild deterministically*, which would
re-pay the embedding cost on every run.

One catch to design around: you **will** want to change `chunk_size` or `k` later in this notebook.
A cache that ignores that is worse than no cache, because it hands you stale vectors with a
confident face.

In [5]:
# [0.2b] TODO: build_docsearch(rebuild: bool = False) -> Chroma
# - use a named collection (COLLECTION below) so it can be addressed and deleted deliberately
# - if rebuild is True, drop the existing collection first
# - if it already holds exactly len(chunks) documents, return it as-is without re-embedding
# HINT: Chroma(collection_name=..., embedding_function=embeddings) opens an existing collection
#       without writing; ._collection.count() tells you what's in it; .delete_collection() drops it
# HINT: set rebuild=True yourself whenever you change chunk_size / the splitter

COLLECTION = "c2_01_policies"


def build_docsearch(rebuild: bool = False) -> Chroma:
    """Return a Chroma store over `chunks`, building it only if it isn't already there."""
    vector_store = Chroma(
        collection_name=COLLECTION,
        embedding_function=embeddings,
    )
    if not rebuild and vector_store._collection.count() == len(chunks):
        return vector_store
    else:
        vector_store.delete_collection()
        vector_store = Chroma(
            collection_name=COLLECTION,
            embedding_function=embeddings,
        )
        vector_store.add_documents(chunks)
        return vector_store

In [6]:
# [0.2c] Given: proof it worked. Run this cell twice — the count must not move.
docsearch = build_docsearch()
retriever = docsearch.as_retriever()

assert docsearch._collection.count() == len(chunks), (
    f"expected {len(chunks)} docs, found {docsearch._collection.count()} — "
    "the store is accumulating; re-check build_docsearch"
)
print(
    f"OK — {docsearch._collection.count()} docs, retriever k={retriever.search_kwargs.get('k', 4)}"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


OK — 16 docs, retriever k=4


In [7]:
# [0.2d] Sanity check before building anything on top of it.
retriever.invoke("mobile phone policy")[0].page_content[:300]

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


'4.\tMobile Phone Policy'

<!-- [A-a] -->
---

# Theme A — how do N chunks become one answer?

The retriever hands back k chunks. The LLM produces one answer. **The step in between has a name,
and it has options.**

The original passes `chain_type="stuff"` without comment. Three alternatives exist:

| chain_type | what it does | LLM calls |
|---|---|---|
| `stuff` | concatenate all chunks into one prompt | 1 |
| `map_reduce` | answer from each chunk separately, then combine the answers | N + 1 |
| `refine` | answer from chunk 1, then revise it chunk by chunk | N |

They differ in cost, in latency, and in what they do when the chunks disagree or don't fit in the
context window. Right now you have no basis for choosing between them — that's the point of the
exercise.

<!-- [A.1a] -->
### A.1 — a factory, so the three are comparable

Write one function that builds a `RetrievalQA` for a given `chain_type`, so the only thing varying
between the three runs is the strategy.

In [8]:
# [A.1b] TODO: build_qa(chain_type: str) -> RetrievalQA
# - use RetrievalQA.from_chain_type
# - llm=llm, retriever=retriever
# - return_source_documents=True  (theme C needs it later, and it costs nothing now)


def build_qa(chain_type: str) -> RetrievalQA:
    """Build a RetrievalQA over `retriever` using the given chain_type."""
    qa = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, return_source_documents=True, chain_type=chain_type)
    return qa

In [9]:
# [A.1c] TODO: test it standalone before comparing anything — build one, invoke it, look at the keys
# of what comes back. What does return_source_documents add to the result dict? #Answer:it added a new key called source_documents
query_1 = "mobile phone policy"

qa = build_qa("stuff")
result = qa.invoke(query_1)

In [10]:
type(result)
result.keys()
len(result["source_documents"])

[i.page_content for i in result["source_documents"]]

dict

dict_keys(['query', 'result', 'source_documents'])

4

['4.\tMobile Phone Policy',
 'The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that employees utilize mobile phones in a manner consistent with company values and legal compliance.\nAcceptable Use: Mobile devices are primarily intended for work-related tasks. Limited personal usage is allowed, provided it does not disrupt work obligations.\nSecurity: Safeguard your mobile device and access credentials. Exercise caution when downloading apps or clicking links from unfamiliar sources. Promptly report security concerns or suspicious activities related to your mobile device.\nConfidentiality: Avoid transmitting sensitive company information via unsecured messaging apps or emails. Be discreet when discussing company matters in public spaces.\nCost Management: Keep personal phone usage separate from company accounts and reimburse the company for any 

<!-- [A.2a] -->
### A.2 — run the same question three ways

Pick a question the document genuinely answers, and send it through all three strategies.

In [11]:
# [A.2b] TODO: for each of "stuff", "map_reduce", "refine":
#   - build the chain, invoke it with the SAME question
#   - print the chain_type and the answer
# Read the three answers side by side before moving on. Are they different? How?
#
# #Answer: stuff and map_reduce both answer the question and agree in substance — same policy,
# same key points, different wording. Nothing distinguishes them at this size.
#
# refine does not answer the question at all. It returns commentary ABOUT the previous answer:
# "the original answer regarding the Mobile Phone Policy remains relevant and comprehensive; the
# context provided about <some other policy> does not directly refine it." That is refine's prompt
# showing through. Every step after the first asks the model to revise an existing answer given one
# more chunk, so when that chunk is about a different policy the model narrates its decision not to
# revise, instead of re-stating the answer. The last chunk processed decides what you are shown,
# and for a question this specific the last chunk is usually irrelevant to it.
#
# Note what is stable and what is not. The meta-commentary is refine's structure and appears every
# run. Which policy it name-drops, and how long each of the three answers is, change on every run
# at temperature=0.5 — so they are not findings and are not recorded here.

question = "What is the mobile phone policy?"

In [12]:
qa_stuff = build_qa("stuff")
stuff = qa_stuff.invoke(question)


In [13]:
qa_map_reduce = build_qa("map_reduce")
map_reduce = qa_map_reduce.invoke(question)

In [14]:
qa_refine = build_qa("refine")
refine = qa_refine.invoke(question)

In [15]:
stuff["result"][:200]
map_reduce["result"][:200]
refine["result"][:200]

'The Mobile Phone Policy sets forth the standards and expectations for the appropriate and responsible usage of mobile devices within the organization. Its purpose is to ensure that employees use mobil'

'The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that em'

'The original answer regarding the Mobile Phone Policy remains relevant and comprehensive, and the context provided about the Smoking Policy does not directly relate to mobile phone usage. Therefore, I'

<!-- [A.3a] -->
### A.3 — now the part that makes it a decision

Three similar answers tell you nothing about which to use. **Cost does.** Count the LLM calls each
strategy actually makes.

A `BaseCallbackHandler` counting `on_llm_start` is the cheapest way to see it.

In [16]:
# [A.3b] TODO: define a callback handler that counts LLM starts
# HINT: subclass BaseCallbackHandler, override on_llm_start(self, serialized, prompts, **kwargs),
#       and increment a counter (remember: one call can carry several prompts)


class CallCounter(BaseCallbackHandler):
    """Counts how many times the LLM is invoked during a chain run.

    Also records each call's finish_reason. A reason of "length" means the model hit max_tokens
    and stopped mid-sentence. Without it, a short answer and a decapitated answer are the same
    number, and every comparison downstream inherits the ambiguity.
    """

    def __init__(self) -> None:
        self.starts = 0  # on_llm_start events
        self.prompts = 0  # prompts carried across those events
        self.finish_reasons = []  # one per generation, in call order

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.starts += 1
        self.prompts += len(prompts)

    def on_llm_end(self, response, **kwargs):
        for generations in response.generations:
            for generation in generations:
                info = generation.generation_info or {}
                reason = info.get("finish_reason")
                if reason is None:
                    message = getattr(generation, "message", None)
                    reason = (getattr(message, "response_metadata", None) or {}).get("finish_reason")
                if reason:
                    self.finish_reasons.append(reason)

    @property
    def truncated(self) -> int:
        """How many calls in this run were cut off by max_tokens."""
        return sum(reason == "length" for reason in self.finish_reasons)


In [17]:
# [A.3c] TODO: re-run the three strategies, this time passing config={"callbacks": [counter]}
# Print a small table: chain_type | n_llm_calls | len(answer)
#
# #Answer: the call counts are exact and deterministic — 1 for stuff, k+1 for map_reduce, k for
# refine — so they are asserted below instead of written down. A re-run that breaks the cost model
# now fails loudly, rather than silently disagreeing with a comment nobody re-reads.
#
# len(answer) is NOT stable. Across runs of this same cell, at temperature=0.5, map_reduce has come
# back at 1528, 1375 and 699 chars and refine at 450, 2811 and 3006. It is printed to be read, never
# recorded as a finding. Read `cut` alongside it: that is the number of calls that stopped because
# they hit max_tokens (512, set in [0b]) rather than because they were finished.

k = retriever.search_kwargs.get("k", 4)
expected_calls = {"stuff": 1, "map_reduce": k + 1, "refine": k}

rows = []
for chain_type in ("stuff", "map_reduce", "refine"):
    counter = CallCounter()
    result = build_qa(chain_type).invoke(question, config={"callbacks": [counter]})
    assert counter.starts == expected_calls[chain_type], (
        f"{chain_type}: expected {expected_calls[chain_type]} LLM calls at k={k}, "
        f"got {counter.starts} — the cost model in [A-a] no longer holds"
    )
    rows.append((chain_type, counter.starts, counter.prompts, len(result["result"]), counter.truncated))

print(f"{'chain_type':<12}{'starts':>8}{'prompts':>9}{'len(answer)':>13}{'cut':>6}")
for chain_type, starts, prompts, n_chars, cut in rows:
    print(f"{chain_type:<12}{starts:>8}{prompts:>9}{n_chars:>13}{cut:>6}")


chain_type    starts  prompts  len(answer)   cut
stuff              1        1         1830     0
map_reduce         5        5         1796     0
refine             4        4         2889     0


<!-- [A-what-a] -->
### What to take away

> Does the call count match the table above (1 / N+1 / N)?
> If `stuff` gives a comparable answer for 1 call instead of 5, when would you ever pay for the
> others? Write your answer down before moving on — theme C is where it stops being hypothetical.

#Answer: no i wouldnt pay for the others. Seems stuff is more worthy than the others

---

# Theme B — the follow-up problem

Two turns. The second one only makes sense if you remember the first.

In [18]:
# [B-a] Given: the trap. Run it and read the second answer carefully.
qa = build_qa("stuff")

turn_1 = qa.invoke("What is the mobile phone policy?")
print("Q1:", turn_1["result"][:400], "\n")

turn_2 = qa.invoke("What can't I do in it?")
print("Q2:", turn_2["result"][:400])

Q1: The Mobile Phone Policy sets forth the standards and expectations for the appropriate and responsible usage of mobile devices within the organization. Its purpose is to ensure that employees use mobile phones in a manner consistent with company values and legal compliance. Key points of the policy include:

- **Acceptable Use**: Mobile devices are primarily for work-related tasks, with limited per 

Q2: In the context of the Internet and Email Policy, you cannot:

1. Use company-provided internet and email services for non-job-related tasks during work hours.
2. Share your login credentials or passwords with others.
3. Open email attachments or click on links from unknown sources without caution.
4. Transmit confidential information without applying encryption.
5. Engage in harassment, discrimina


<!-- [B.1a] -->
### B.1 — diagnose before fixing

The second answer is wrong, or vague, or about the wrong policy. **Don't accept the obvious
explanation without checking it.** The claim is: the retriever never saw the word "mobile", so it
retrieved on "what can't I do", which matches half the document.

Prove it.

In [19]:
# [B.1b] TODO: retrieve directly for the follow-up question (no chain, just retriever.invoke)
# and print the first ~200 chars of each returned chunk.
# Which policies came back? Is "mobile" among them? #Answer: 4 policies came back: internet, mobile, smoking drugs. yes, mobile is among them.

docsearch = build_docsearch()
retriever = docsearch.as_retriever()
result = retriever.invoke("What can't I do in it?")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [20]:
type(result)
len(result)
result[0].page_content[:200]
result[1].page_content[:200]
result[2].page_content[:200]
result[3].page_content[:200]

list

4

'Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and '

'The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that em'

'Policy Purpose: The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on company premises. This policy is in place to ensure a safe and healthy environm'

'Policy Objective: The Drug and Alcohol Policy is established to establish clear expectations and guidelines for the responsible use of drugs and alcohol within the organization. This policy aims to ma'

<!-- [B.2a] -->
### B.2 — the fix, and the thing worth looking at

`ConversationalRetrievalChain` inserts a step before retrieval: it sends the chat history plus the
new question to the LLM and asks for a **standalone question** — "What can't I do in it?" becomes
something like "What are the restrictions in the company mobile phone policy?"

That rewritten question is the whole mechanism, and it's normally invisible.
`return_generated_question=True` exposes it.

Per 0.1, memory is the state that must **reset**, not cache. Every experiment below starts from
turn zero — otherwise the condensed question is computed against a history containing your previous
three attempts, and you'll be debugging your own leftovers.

In [21]:
# [B.2b]

# TODO: fresh_memory() -> ConversationBufferMemory
# - memory_key="chat_history", return_messages=True
# - a NEW instance every call — that is the entire point
# HINT: the kwarg is return_messageS. The original IBM notebook writes return_message,
#       which silently does nothing.
def fresh_memory() -> ConversationBufferMemory:
    """A conversation memory with no history. Call at the start of every experiment."""
    return ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer" )


# TODO: build_conv_qa() -> ConversationalRetrievalChain
def build_conv_qa() -> ConversationalRetrievalChain:
    return ConversationalRetrievalChain.from_llm(
        llm=llm, retriever=retriever, memory=fresh_memory(), return_generated_question=True
    )


# - building it should imply a fresh conversation, so call fresh_memory() inside

In [22]:
# [B.2c] TODO: run the same two turns through a freshly built chain.
# For each turn print BOTH result["answer"] and result["generated_question"].
# Turn 1's generated question should be boring. Turn 2's is the interesting one.

# Then, to prove the reset matters: run this whole cell a second time. The generated question for
# turn 2 should be IDENTICAL to the first run. If it drifts, memory is leaking between runs.
#
# #Answer: turn 1's question is passed through unchanged — with an empty history there is nothing to
# condense. Turn 2's is rewritten from "What can't I do in it?" into a standalone question that
# names the phone policy. That rewrite is the entire mechanism.
#
# The wording of it moves between runs ("What actions are prohibited under/by the mobile phone
# policy?"), so it is not transcribed here. The property that must hold every run is asserted
# instead: turn 2's condensed question still names the phone. Building the chain builds a fresh
# memory, so a second run of this cell starts from turn zero and the assert holds again — a leaking
# memory would condense turn 2 against the previous run's history and lose the referent.

conv_qa = build_conv_qa()
turns = ["What is the mobile phone policy?", "What can't I do in it?"]

for turn in turns:
    result = conv_qa.invoke({"question": turn})
    print(f"Q  {turn}")
    print(f"Q' {result['generated_question']}")
    print(f"A  {result['answer'][:300]}\n")

assert "phone" in result["generated_question"].lower(), (
    f"the follow-up was not resolved to the phone policy: {result['generated_question']!r} — "
    "history is not reaching the condense step"
)


Q  What is the mobile phone policy?
Q' What is the mobile phone policy?
A  The Mobile Phone Policy sets forth the standards and expectations for the appropriate and responsible usage of mobile devices within the organization. Its purpose is to ensure that employees use mobile phones in a manner consistent with company values and legal compliance. Key points of the policy i

Q  What can't I do in it?
Q' What actions are prohibited under the mobile phone policy?
A  The following actions are prohibited under the Mobile Phone Policy:

1. Transmitting sensitive company information via unsecured messaging apps or emails.
2. Discussing company matters in public spaces in a manner that could compromise confidentiality.
3. Failing to safeguard mobile devices and acce



<!-- [B.3a] -->
### B.3 — the comparison that only you can make

C1 solved this same problem a completely different way: no condensation step at all. The
checkpointer replayed the conversation, and the *agent* decided what to pass to `get_context`.

> Two mechanisms, same symptom cured:
> - condensation — one extra LLM call, always, whether or not the question needs it
> - agent reasoning — no extra call, but the query is chosen by a model that could choose badly
>
> Which one fails more gracefully? Which would you rather debug at 2am?
> When does the condensation step actively *hurt* — think about a follow-up that isn't a
> follow-up at all.

---

# Theme C — can this thing actually summarize?

The original notebook is titled *Summarize Private Documents* and asks
*"Can you summarize the document for me?"* through a `k=4` retriever.

Before running it: **establish ground truth.** You can read this corpus.

In [23]:
conv_qa = build_conv_qa()
questions = ["What are the 9 company policies in this document?"]
for i in questions:
    result = conv_qa.invoke({"question": i})
    print(result["answer"])
    result["generated_question"]  

I don't know.


'What are the 9 company policies in this document?'

In [24]:
# [C-a] TODO: how many distinct policies are in companyPolicies.txt? #Answer: 9 policies
# Read the raw file (or skim `documents[0].page_content`) and list their names.
# This is the answer key for everything below.
# print(documents[0].page_content)

#Answer:
POLICIES = ["Code of Conduct", "Recruitment Policy", "Internet and Email Policy", "Mobile Phone Policy", "Smoking Policy", "Drug and Alcohol Policy", "Health and Safety Policy", "Anti-discrimination and Harassment Policy" , "Discipline and Termination Policy"]

In [25]:
# [C-b] Given: scoring against the answer key, so no summary is counted by hand.
#
# A policy counts as covered when its full name from POLICIES appears in the text. Match on the
# full name, never on a single word — these policies quote each other's vocabulary. "harassment"
# and "discrimination" both appear inside the Recruitment and Internet-and-Email sections, and
# "safety" is one of the Code of Conduct's listed principles. Word matching scores 7/9 on a summary
# that genuinely covers 4.

SUMMARIES = {}  # label -> (text, counter), filled by record() below


def coverage(text: str) -> tuple[int, list[str]]:
    """Return (n_covered, missing_names) for a summary, scored against POLICIES."""
    lowered = text.lower()
    missing = [policy for policy in POLICIES if policy.lower() not in lowered]
    return len(POLICIES) - len(missing), missing


def print_row(label, text, counter=None) -> None:
    n_covered, missing = coverage(text)
    cost = f"{counter.starts} calls" if counter is not None else "-"
    cut = "  <- TRUNCATED at max_tokens" if counter is not None and counter.truncated else ""
    print(f"{label:<12}{n_covered:>3}/{len(POLICIES):<6}{cost:>9}{cut}")
    if missing:
        print(f"{'':12}   missing: {', '.join(missing)}")


def record(label, text, counter=None) -> None:
    """Score one summary, print its row, and keep it for the final comparison."""
    SUMMARIES[label] = (text, counter)
    print_row(label, text, counter)


def comparison() -> None:
    """Every approach recorded so far, on the same two axes."""
    print(f"{'approach':<12}{'coverage':>9}{'cost':>9}")
    for label, (text, counter) in SUMMARIES.items():
        print_row(label, text, counter)


<!-- [C.1a] -->
### C.1 — ask for a summary through retrieval

In [26]:
# [C.1b] TODO: ask build_qa("stuff") to summarize the document.
# Then: how many of the policies you listed above appear in the answer?
# And separately — how many chunks did it actually see? (result["source_documents"])
#
# #Answer: it sees k chunks out of len(chunks), and covers roughly a third of the policies. The
# fraction is computed below rather than written here, because it moves between runs. Three things
# that do not move, and matter more than the fraction:
#
# - Retrieved is not the same as covered. Chunks come back that never reach the answer, and a
#   policy can reach the answer from inside a chunk whose heading names a different policy. Score
#   the answer, never source_documents.
# - Some retrieved chunks are bare section headings with no body ("5.\tSmoking Policy"), so k=4
#   buys fewer than four chunks of actual content.
# - None of this is a prompt problem. The model cannot summarize text it was never handed.

counter = CallCounter()
qa = build_qa("stuff")
result = qa.invoke("summarize the document", config={"callbacks": [counter]})

print(f"saw {len(result['source_documents'])} of {len(chunks)} chunks:")
for n, doc in enumerate(result["source_documents"]):
    print(f"  [{n}] {doc.page_content[:90]!r}")
print()
record("retrieval", result["result"], counter)


saw 4 of 16 chunks:
  [0] '9.\tDiscipline and Termination Policy'
  [1] '7.\tHealth and Safety Policy\n\nOur commitment to health and safety is paramount. We prioriti'
  [2] "The Discipline and Termination Policy underscores the organization's commitment to maintai"
  [3] '5.\tSmoking Policy'

retrieval     2/9       1 calls
               missing: Code of Conduct, Recruitment Policy, Internet and Email Policy, Mobile Phone Policy, Smoking Policy, Drug and Alcohol Policy, Anti-discrimination and Harassment Policy


In [27]:
result["result"]
len(result["source_documents"])
result["source_documents"][0].page_content[:150]
result["source_documents"][1].page_content[:150]
result["source_documents"][2].page_content[:150]
result["source_documents"][3].page_content[:150]

'The document outlines several key policies within an organization to ensure a safe, respectful, and productive work environment. \n\n1. **Health and Safety Policy**: The organization prioritizes the well-being of employees, customers, and the public by complying with health and safety laws, maintaining a hazard-free workplace, providing training, and encouraging communication about safety concerns.\n\n2. **Discipline and Termination Policy**: This policy emphasizes the importance of performance and conduct expectations for all personnel. It details the process for disciplinary actions, which may include warnings or suspension, and outlines the conditions under which termination may occur, such as persistent performance issues or policy violations. The policy also describes the termination procedure, including fairness and legal compliance, and the exit process for departing employees.\n\nOverall, the document serves as a framework for maintaining standards in health and safety, discip

4

'9.\tDiscipline and Termination Policy'

'7.\tHealth and Safety Policy\n\nOur commitment to health and safety is paramount. We prioritize the well-being of our employees, customers, and the publi'

"The Discipline and Termination Policy underscores the organization's commitment to maintaining a productive, ethical, and respectful work environment."

'5.\tSmoking Policy'

<!-- [C.2a] -->
### C.2 — name the gap

> The retriever returned k chunks out of the total. A summary of k chunks is not a summary of the
> document — it's a summary of whatever happened to embed near the word "summarize".
>
> Notice this is not a prompt problem. No amount of prompt engineering makes the model summarize
> text it was never given. **Retrieval answers questions; summarization needs the whole corpus.**
> They are different operations that happen to share a pipeline.

<!-- [C.3a] -->
### C.3 — do it properly

Drop the retriever entirely. `load_summarize_chain` runs over *all* chunks — and now
`map_reduce` / `refine` from theme A stop being an academic choice, because "stuff the whole
document into one prompt" is exactly what stops working as documents grow.

In [28]:
# [C.3b] TODO: summarize ALL chunks with load_summarize_chain(llm, chain_type="map_reduce")
# - it takes documents directly: chain.invoke({"input_documents": chunks})
# - count the policies in this answer, against the same answer key
# - count the LLM calls with your CallCounter
#
# #Answer: full coverage, for len(chunks) + 1 calls — one per chunk, plus one to combine them. The
# call count is deterministic, so it is asserted; the coverage is computed. Worth noticing that the
# reduce step's answer stays short and never approaches max_tokens: each chunk is summarized
# independently and only the summaries are combined, so the final answer's length does not grow
# with the corpus. That is exactly what refine cannot do — see [C.3c].

counter = CallCounter()
summarize = load_summarize_chain(llm=llm, chain_type="map_reduce")
result = summarize.invoke({"input_documents": chunks}, config={"callbacks": [counter]})

assert counter.starts == len(chunks) + 1, (
    f"expected {len(chunks) + 1} calls (one per chunk + one reduce), got {counter.starts}"
)
print(result["output_text"], "\n")
record("map_reduce", result["output_text"], counter)


The "Code of Conduct" establishes principles of integrity, respect, accountability, and compliance within the organization, promoting ethical behavior and a positive work environment. The Recruitment Policy ensures fair and transparent hiring practices, emphasizing diversity and inclusion. The Internet and Email Policy governs the responsible use of digital resources, while the Mobile Phone Policy outlines acceptable mobile device usage. The Smoking Policy promotes a smoke-free environment, and the Drug and Alcohol Policy prohibits substance misuse on company premises. The Health and Safety Policy emphasizes workplace safety and well-being, while the Anti-Discrimination and Harassment Policy fosters a respectful, inclusive environment. Finally, the Discipline and Termination Policy details procedures for addressing misconduct and performance issues, ensuring fairness and consistency in disciplinary actions. Each policy is regularly reviewed to align with legal standards and best practi

In [29]:
type(result)
result.keys()
type(result["input_documents"])
len(result["input_documents"])

[i.page_content[:100] for i in result["input_documents"]]

dict

dict_keys(['input_documents', 'output_text'])

list

16

['1.\tCode of Conduct',
 'Our Code of Conduct outlines the fundamental principles and ethical standards that guide every membe',
 '2.\tRecruitment Policy',
 'Our Recruitment Policy reflects our commitment to attracting, selecting, and onboarding the most qua',
 '3.\tInternet and Email Policy',
 'Our Internet and Email Policy is established to guide the responsible and secure use of these essent',
 '4.\tMobile Phone Policy',
 'The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and resp',
 '5.\tSmoking Policy',
 'Policy Purpose: The Smoking Policy has been established to provide clear guidance and expectations c',
 '6.\tDrug and Alcohol Policy',
 'Policy Objective: The Drug and Alcohol Policy is established to establish clear expectations and gui',
 '7.\tHealth and Safety Policy\n\nOur commitment to health and safety is paramount. We prioritize the wel',
 'The Anti-Discrimination and Harassment Policy is a testament to the commitment of this organ

In [30]:
# [C.3c] TODO: same thing with chain_type="refine". Compare all three summaries:
#   retrieval-based | map_reduce | refine
# on two axes: coverage (policies mentioned / total) and cost (LLM calls).
#
# #Answer: the table is built below from the three recorded runs rather than copied out, because
# refine's coverage moves between runs. The ranking does not: retrieval covers the least for one
# call, map_reduce covers everything for len(chunks) + 1, refine lands in between for len(chunks).
#
# Read refine's row with the TRUNCATED flag, not the fraction. refine carries ONE growing answer and
# re-emits all of it at every step, so once that answer reaches max_tokens (512, set in [0b]) every
# later step is cut short and the end of the document can never be added. The policies it loses are
# always the last ones in document order — measured at exactly 512 tokens on three separate runs.
# map_reduce escapes this because each map output is short and independent; its reduce step came
# back around 220-280 tokens, nowhere near the ceiling.
#
# So refine's number here is a measurement of the token cap, not of the strategy. To see it move,
# raise max_tokens in [0b] and re-run this cell alone. That distinction is the whole reason the
# counter records finish_reason: coverage alone cannot tell "covered less" from "was cut off".

counter = CallCounter()
summarize = load_summarize_chain(llm=llm, chain_type="refine")
result = summarize.invoke({"input_documents": chunks}, config={"callbacks": [counter]})

assert counter.starts == len(chunks), (
    f"expected {len(chunks)} calls (one per chunk), got {counter.starts}"
)
print(result["output_text"], "\n")
record("refine", result["output_text"], counter)

print("\n--- all three, same axes ---")
comparison()


The "Code of Conduct" outlines the fundamental principles and ethical standards expected of individuals within our organization, emphasizing integrity, respect, accountability, safety, and environmental responsibility. It serves as a guideline for decision-making and interactions, promoting a positive and inclusive work environment where diversity is embraced and contributions are valued. Our Recruitment Policy aligns with these principles, focusing on attracting, selecting, and onboarding qualified and diverse candidates. As an equal opportunity employer, we do not discriminate based on protected statuses and actively promote diversity and inclusion. We maintain transparency in our recruitment processes, ensuring clear job descriptions and objective selection criteria. Additionally, we prioritize data privacy, provide timely feedback to candidates, and offer comprehensive onboarding for new employees. This policy is essential for building a talented workforce that reflects our commitm

In [31]:
type(result)
result.keys()
type(result["input_documents"])
len(result["input_documents"])

[i.page_content[:100] for i in result["input_documents"]]
# result["input_documents"]


dict

dict_keys(['input_documents', 'output_text'])

list

16

['1.\tCode of Conduct',
 'Our Code of Conduct outlines the fundamental principles and ethical standards that guide every membe',
 '2.\tRecruitment Policy',
 'Our Recruitment Policy reflects our commitment to attracting, selecting, and onboarding the most qua',
 '3.\tInternet and Email Policy',
 'Our Internet and Email Policy is established to guide the responsible and secure use of these essent',
 '4.\tMobile Phone Policy',
 'The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and resp',
 '5.\tSmoking Policy',
 'Policy Purpose: The Smoking Policy has been established to provide clear guidance and expectations c',
 '6.\tDrug and Alcohol Policy',
 'Policy Objective: The Drug and Alcohol Policy is established to establish clear expectations and gui',
 '7.\tHealth and Safety Policy\n\nOur commitment to health and safety is paramount. We prioritize the wel',
 'The Anti-Discrimination and Harassment Policy is a testament to the commitment of this organ

<!-- [C-what-a] -->
### What this sets up

> You now have a concrete case where `stuff` is not merely cheaper but *wrong*, and where the
> theme A choice has a measurable right answer. Coverage is a crude metric — but it is a metric,
> which is more than "the answer reads fine" gives you. That's the seed of the eval harness
> (`C2-analysis.md`, cross-cutting section).

LlamaIndex names this distinction in its API — a `SummaryIndex` traverses everything while a
`VectorStoreIndex` retrieves, and `tree_summarize` is `map_reduce` under a different name. That
comparison is theme E, in `C2-02`.

---

## Where this lands

- **Themes A, B, C** — done here, in a notebook, because they're ideas that need values on screen.
- **Theme D** (`src/ibm_rag_and_agentic_ai/c2/`) — the application layer: entrypoints, runtime
  ingestion, session state. Not teachable in a notebook, which is the whole reason it's a module.
- **Open loop from C1** — `quoted_excerpts` / source attribution. `return_source_documents=True`
  has been on every chain in this notebook; that's the raw material for closing it.

# I have hand written some answers that lost its grounding at every. So the AI did some changes below

1. Deterministic facts → assert, not a comment. Call counts don't vary: stuff is 1, map_reduce is k+1, refine is k. Asserting them means a re-run that breaks the claim fails loudly, instead of silently disagreeing with a comment nobody re-reads. That's strictly better than transcribing, and it's the same move [0.2c] already makes for the chunk count.

2. Varying measurements → compute and print. One helper against POLICIES — something like coverage(text) -> (hits, missing) — called in [C.1b], [C.3b], [C.3c]. The cell's own output becomes the record, and it's re-derived every run by construction. Nothing to keep in sync because nothing is duplicated.

3. Judgment → prose, phrased so no number appears in it. That's where your answers belong, and most of yours are already there.

In [ ]:
# cell	change
# [A.3c]	assert the 1 / k+1 / k call counts; delete the transcribed length table — the cell prints it; add one prose line that length swings run to run while call count doesn't
# [C.1b] [C.3b] [C.3c]	call the coverage helper; keep the prose interpretation, drop the fractions
# [B.2c]	don't quote the generated question — assert "mobile" in generated_question.lower(). That is your claim ("clearly pointing mobile"), and it's the one thing that must hold for condensation to have worked
# [A.2b]	rephrase to the stable observation: refine returned commentary about the previous answer instead of an answer. Which policy it name-dropped is run-specific noise
